In [ ]:
!ls -lh kojo_model.pt
!pip install -q fastapi uvicorn ultralytics pillow python-multipart pyngrok

ls: cannot access 'kojo_model.pt': No such file or directory


In [ ]:
!pwd
!ls -la
!find /content -name "*.pt" 2>/dev/null

/content
total 16
drwxr-xr-x 1 root root 4096 Oct 21 13:28 .
drwxr-xr-x 1 root root 4096 Nov 22 15:07 ..
drwxr-xr-x 4 root root 4096 Oct 21 13:28 .config
drwxr-xr-x 1 root root 4096 Oct 21 13:28 sample_data


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving kojo_model.pt to kojo_model.pt


In [ ]:
!ls -lh kojo_model.pt

-rw-r--r-- 1 root root 5.3M Nov 22 15:11 kojo_model.pt


In [ ]:
%%writefile main.py
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from ultralytics import YOLO
from PIL import Image
import io

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

model = YOLO("kojo_model.pt")

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    results = model.predict(source=image, conf=0.4)[0]
    detections = []

    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        class_id = int(box.cls[0])
        detections.append({
            "label": results.names[class_id],
            "confidence": round(float(box.conf[0]), 2),
            "bbox": [x1, y1, x2, y2]
        })

    return {"detections": detections}

Writing main.py


In [ ]:
import subprocess, time
from pyngrok import ngrok

# If ngrok asks for an auth token, uncomment and paste yours from dashboard.ngrok.com:
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

proc = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

public_url = ngrok.connect(8000).public_url
print("Public API URL:", public_url)
print("Swagger Docs:", f"{public_url}/docs")

ERROR:pyngrok.process.ngrok:t=2026-11-22T15:15:41+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-11-22T15:15:41+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-11-22T15:15:41+0000 lvl=crit msg="command failed" err="authentication failed: This ngrok session is not authenticated. ngrok requi

PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
!ls -lh kojo_model.pt

-rw-r--r-- 1 root root 5.3M Nov 22 15:11 kojo_model.pt


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess, time, re

subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(4)
for _ in range(40):
    line = tunnel.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print("COPY THIS LINK:", url)
        print("TEST IT HERE:", f"{url}/docs")
        break

COPY THIS LINK: https://description-veterinary-married-manufacturing.trycloudflare.com
TEST IT HERE: https://description-veterinary-married-manufacturing.trycloudflare.com/docs


In [ ]:
!ls -lh kojo_model.pt

ls: cannot access 'kojo_model.pt': No such file or directory


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving kojo_model.pt to kojo_model.pt


In [ ]:
!ls -lh kojo_model.pt
!pip install -q fastapi uvicorn ultralytics pillow python-multipart

-rw-r--r-- 1 root root 5.3M Nov 22 16:36 kojo_model.pt


In [ ]:
%%writefile main.py
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from ultralytics import YOLO
from PIL import Image
import io

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

model = YOLO("kojo_model.pt")

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    results = model.predict(source=image, conf=0.4)[0]
    detections = []

    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        class_id = int(box.cls[0])
        detections.append({
            "label": results.names[class_id],
            "confidence": round(float(box.conf[0]), 2),
            "bbox": [x1, y1, x2, y2]
        })

    return {"detections": detections}

Writing main.py


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess, time, re

subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(4)
for _ in range(40):
    line = tunnel.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print("NEW PUBLIC URL:", url)
        print("TEST DOCS:", f"{url}/docs")
        break

NEW PUBLIC URL: https://asia-guests-arms-violin.trycloudflare.com
TEST DOCS: https://asia-guests-arms-violin.trycloudflare.com/docs
